# exp5 — leakage-safe candidate feature tables

**Timestamp:** 2026-10-24 14:20 MSK  
**Run group:** `exp5-feature-build`  
**Intent:** Materialize the requested user-show history, user, channel, time, schedule, metadata and collaborative features before any reranker training.

## Method / leakage boundary

The feature builder reads only `data/exp5_user_airings_rich.parquet` rows with `week <= train_end` for features, and uses only the target-window schedule for future schedule context. Target-window views are loaded in a separate frame and merged only at the end as `label_050` / `label_080`. The early split is history≤16→target17–20; main is history≤20→target21–30. Candidate rows come from expanded9 tripartite candidate generation, including the four schedule-propensity aggregates.

Builder: `artifacts/scripts/exp5_build_features.py`. Outputs: `data/exp5_features_early.parquet` and `data/exp5_features_main.parquet`.


In [1]:
! .venv/bin/python artifacts/scripts/exp5_build_features.py early

[exp5] split=early; output=/Users/davidkk/Kaggle_Case/data/exp5_features_early.parquet
[exp5] rows=2,174,364; columns=101; users=1,218
[exp5] labels .5 positives=27,947; .8 positives=16,478
[exp5] duplicate user-show rows=0; target cutoff=20; history cutoff=16

In [2]:
! .venv/bin/python artifacts/scripts/exp5_build_features.py main

[exp5] split=main; output=/Users/davidkk/Kaggle_Case/data/exp5_features_main.parquet
[exp5] rows=2,605,151; columns=101; users=1,244
[exp5] labels .5 positives=54,477; .8 positives=35,451
[exp5] duplicate user-show rows=0; target cutoff=30; history cutoff=20

In [3]:
import pandas as pd
required=['freq_watch_pct_030','freq_watch_pct_050','freq_watch_pct_080','freq_1w','freq_2w','freq_4w','freq_8w','rank_freq','rank_recent','days_since_last_watch','distinct_weeks_watched','sum_watch_seconds','mean_watch_pct','max_watch_pct','median_watch_pct','total_views','total_seconds','active_days','active_weeks','mean_session_duration','channel_views','channel_seconds','channel_share','recent_channel_share','user_dow_affinity','user_hour_affinity','user_slot_affinity','user_channel_dow_affinity','user_channel_hour_affinity','user_channel_dow_slot_affinity','future_airings_count','future_total_airtime','past_airings_count','past_total_airtime','future_past_airtime_ratio','mean_future_hour','schedule_hour_shift','channel_id','tv_show_category','tv_show_genre_1','tv_show_genre_2','tv_show_genre_3','year_of_production','lightfm_score','lightfm_rank','user_bias','item_bias','ppr_score','ppr_rank','als_score','als_rank','airing_propensity_max','airing_propensity_mean','airing_propensity_sum','airing_propensity_logsumexp']
for split in ['early','main']:
 d=pd.read_parquet(f'data/exp5_features_{split}.parquet')
 print(f'{split}: shape={d.shape}; users={d.user_id.nunique()}; missing={len(set(required)-set(d.columns))}')
 print(f'  duplicate user-show rows={d.duplicated(["user_id","tv_show_id"]).sum()}; labels .5/.8={d.label_050.sum()}/{d.label_080.sum()}; null labels={d[["label_050","label_080"]].isna().sum().sum()}; label_08_gt_05={(d.label_080>d.label_050).sum()}')

early: shape=(2174364, 101); users=1218; missing=0
  duplicate user-show rows=0; labels .5/.8=27947/16478; null labels=0; label_08_gt_05=0
main: shape=(2605151, 101); users=1244; missing=0
  duplicate user-show rows=0; labels .5/.8=54477/35451; null labels=0; label_08_gt_05=0

## Result interpretation

Both feature tables have the expected 101 columns, all required feature families, unique `(user_id,tv_show_id)` rows, non-null binary labels and `label_080 <= label_050`. The candidate table carries LightFM scores/ranks/biases, tripartite-PPR scores/ranks, ALS scores/ranks, source ranks, and `airing_propensity_{max,mean,sum,logsumexp}`. Target labels are separate columns and are not used by the feature calculations.

The main candidate Recall@500 remains below the user-confirmed 0.80 gate. **No binary LightGBM model or LGBMRanker has been trained.** This notebook records feature construction and validation only; it makes no MAP@5 or Kaggle score claim.

The initial early run emitted pandas deprecation warnings for `inplace`; the builder was corrected before the main run. This did not alter feature values.
